## Analyse des données patients actualisées en octobre 2023 

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import time
import datetime
import plotly.graph_objects as go


In [3]:
#df_clinique=pd.read_csv("../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";")

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_12756\3681731435.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clinique=pd.read_csv("../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";")


### Check for NaN values 

#### Sexe 

In [4]:
# Statistiques descriptive pour le sexe des patients

print("\nStatistiques descriptive pour le sexe des patients:\n")

df_stats_sexe = df_clinique.groupby(["patient_sexe"]).size()
print(df_stats_sexe)

print(f"\nSomme du nombre d'hommes et de femmes :{df_stats_sexe['F']+df_stats_sexe['M']}")

print(f"\nNombre de patients ayant nan comme sexe: {df_clinique['patient_sexe'].isnull().sum()}")

print(f"\nNombre total de patients : {len(df_clinique)} \n")



Statistiques descriptive pour le sexe des patients:

patient_sexe
F    44877
M    13001
dtype: int64

Somme du nombre d'hommes et de femmes :57878

Nombre de patients ayant nan comme sexe: 0

Nombre total de patients : 57878 



In [5]:
df_clinique[df_clinique['patient_sexe']=='UN']

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,...,CODE_DEPT,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib,patho


#### Age 

In [6]:
# Statistiques descriptive pour les dates de naissances des patients
print("\nStatistiques descriptive pour les dates de naissances des patients:\n")

print(f"\nNombre de patients ayant nan comme date de naissance: {df_clinique['date_naissance'].isnull().sum()}")

nan_in_sexe = df_clinique['patient_sexe'].isnull()
un_in_sexe = df_clinique['patient_sexe']=='UN'
nan_in_age = df_clinique['date_naissance'].isnull()
both_nan = df_clinique[nan_in_sexe & nan_in_age ]

print(f"Nombre de patients ayant nan comme date de naissance et comme sexe: {both_nan.shape[0]}")



Statistiques descriptive pour les dates de naissances des patients:


Nombre de patients ayant nan comme date de naissance: 0
Nombre de patients ayant nan comme date de naissance et comme sexe: 0


In [7]:
df_noNa = df_clinique.dropna()

current_year = datetime.date.today().year
df_noNa["annee_naissance"] = df_noNa.date_naissance.str[:4].astype(int)
df_noNa["age"]= current_year - df_noNa["annee_naissance"].astype(int)
df_noNa = df_noNa.drop(["annee_naissance"], axis=1)
df_noNa.head()

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_12756\3163852153.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noNa["annee_naissance"] = df_noNa.date_naissance.str[:4].astype(int)
C:\Users\lpokambo\AppData\Local\Temp\ipykernel_12756\3163852153.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noNa["age"]= current_year - df_noNa["annee_naissance"].astype(int)


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,...,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib,patho,age
0,0,0,0,1,34 RUE DES FRERES CHAUSSONS,92600.0,ASNIERES-SUR-SEINE,34 RUE DES FRERES CHAUSSONS 92600 ASNI...,2.289499,48.916298,...,F,1985-06,paris,32.0,1.0,2018-04-30,C92,Leucémie myéloïde aiguë,Hemato,39
1,1,1,1,2,11 RUE EMILE DUBOIS,75014.0,PARIS,11 RUE EMILE DUBOIS 75014 PARIS,2.336628,48.831707,...,F,1970-10,paris,46.0,1.0,2017-07-17,C69,Tumeur maligne de la choroïde,Ophtalmo,54
2,2,2,2,3,48 CHEMIN VERT,78680.0,EPONE,48 CHEMIN VERT 78680 EPONE,1.797376,48.950412,...,M,1949-03,saint-cloud,69.0,1.0,2019-01-15,C61,Tumeur maligne de la prostate,Uro,75
3,3,3,3,4,18 ALLEE DE LA CHARNILLE,47140.0,SAINT-SYLVESTRE-SUR-LOT,18 ALLEE DE LA CHARNILLE 47140 SAIN...,0.809474,44.404892,...,M,1957-02,paris,62.0,1.0,2019-06-30,C10,Tumeur maligne à localisations contiguës de l ...,ORL,67
4,4,4,4,5,31 RUE DU GENERAL DE MIRIBEL,92500.0,RUEIL-MALMAISON,31 RUE DU GENERAL DE MIRIBEL 92500 RUEI...,2.173326,48.865232,...,M,1941-12,saint-cloud,76.0,1.0,2018-02-22,C61,Tumeur maligne de la prostate,Uro,83


In [8]:
age_interval = [(0,9),(10,19),(20,29),(30,39),(40,49),(50,59),(60,69),(70,79),(80,89),(90,99),(100,150)]
df_noNa["ageInterv"] = pd.cut(df_noNa.age, bins=[interval[0] for interval in age_interval] + [age_interval[-1][1]], labels = ['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99','100+'])
df_pyramidage = df_noNa.groupby(["ageInterv", "patient_sexe"]).size()
df_pyramidage = df_noNa.pivot_table(index="ageInterv", columns="patient_sexe", values="pseudo_provisoire", aggfunc="count", fill_value=0).reset_index()

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_12756\4170278522.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_pyramidage = df_noNa.groupby(["ageInterv", "patient_sexe"]).size()
C:\Users\lpokambo\AppData\Local\Temp\ipykernel_12756\4170278522.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_pyramidage = df_noNa.pivot_table(index="ageInterv", columns="patient_sexe", values="pseudo_provisoire", aggfunc="count", fill_value=0).reset_index()


In [9]:
y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

#### Code CIM10 

In [10]:

# Statistiques descriptive pour le code CIM10 des patients

print("\nStatistiques descriptive pour le code CIM10 des patients:\n")

df_cim10 = df_clinique.groupby(["topo_initiale_cim10", "topo_initialelib"]).size()

fig = px.histogram(df_clinique, x="topo_initiale_cim10").update_xaxes(categoryorder = "total descending")
fig.update_layout(title = "Répartition des codes CIM pour nos données géocodées")
fig.show()

##ajout nom patho 


Statistiques descriptive pour le code CIM10 des patients:



In [ ]:
#df_cim10.to_csv("/home/jovyan/work/canc_air/01_data/data_octobre_2023/codeCIM10.csv")

### Verification na values : 

In [11]:
print(f"Nombre de personne n'ayant pas de code CIM10 : {df_clinique.topo_initiale_cim10.isnull().sum()}")

Nombre de personne n'ayant pas de code CIM10 : 0


In [12]:
## verifiier combien on retire 21 dans 364
print("Est-ce que les 21 personnes n'ayant pas de sexe ni d'age n'ont pas de code CIM10 non plus?")


Est-ce que les 21 personnes n'ayant pas de sexe ni d'age n'ont pas de code CIM10 non plus?


In [13]:
nan_cim10 = df_clinique.topo_initiale_cim10.isna() 
nan_sexe = df_clinique.patient_sexe.isna()

df_clinique[nan_cim10 & nan_sexe]
print("oui")

oui


In [14]:
df_centre = df_clinique.groupby(["centre"]).size()

fig = px.histogram(df_clinique, x="centre").update_xaxes(categoryorder = "total descending")
fig.update_layout(title = "Répartition des patients selon leur centre de provenance")
fig.show()
